# 새 영상 2종 스냅샷 추출 — Tuned 파라미터 적용

각 씬 클립의 **중간 프레임**을 추출하고, 튜닝된 YOLOv8L 파라미터로 object detection 결과를 시각화합니다.

| 파라미터 | 값 |
|----------|----|
| **conf** | 0.25 |
| **iou** | 0.70 |
| **imgsz** | 800 |

**출력 폴더:**
- `snapshots_scenes2/` — 요리 영상 (SampleVideo_Scenes2)
- `snapshots_scenes3/` — 자취방 영상 (SampleVideo_Scenes3)

**실행 전 체크리스트:**
- [ ] 런타임 > 런타임 유형 변경 > **GPU (T4)** 선택
- [ ] Google Drive에 아래 폴더 업로드 확인
  - `내 드라이브/SampleVideo.Test/SampleVideo_Scenes2/` (요리, 179개)
  - `내 드라이브/SampleVideo.Test/SampleVideo_Scenes3/` (자취방, 223개)

## 0. 환경 설정

In [ ]:
# GPU 확인
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
print("GPU 확인 완료!")

In [ ]:
# 패키지 설치
!pip install -q ultralytics
print("ultralytics 설치 완료!")

In [ ]:
# Google Drive 마운트
from google.colab import drive
drive.mount('/content/drive')
print("Drive 마운트 완료!")

## 1. 경로 및 파라미터 설정

In [ ]:
import os

BASE_DRIVE = "/content/drive/MyDrive/SampleVideo.Test"

# 입력 영상 폴더 & 출력 스냅샷 폴더
VIDEO_SETS = {
    "요리":   {
        "input":  os.path.join(BASE_DRIVE, "SampleVideo_Scenes2"),
        "output": "/content/snapshots_scenes2",
        "drive_output": os.path.join(BASE_DRIVE, "snapshots_scenes2"),
    },
    "자취방": {
        "input":  os.path.join(BASE_DRIVE, "SampleVideo_Scenes3"),
        "output": "/content/snapshots_scenes3",
        "drive_output": os.path.join(BASE_DRIVE, "snapshots_scenes3"),
    },
}

# 튜닝 파라미터
CONF   = 0.25
IOU    = 0.70
IMGSZ  = 800
MODEL_PATH = "yolov8l.pt"

# MVP 15 클래스 ID
MVP_CLASS_IDS = [16, 26, 39, 41, 45, 53, 56, 57, 59, 60, 62, 63, 65, 67, 72]

# 출력 폴더 생성
for name, paths in VIDEO_SETS.items():
    os.makedirs(paths["output"], exist_ok=True)
    clips = len([f for f in os.listdir(paths["input"]) if f.endswith('.mp4')])
    print(f"[{name}] 입력: {clips}개 클립 | 출력: {paths['output']}")

## 2. 모델 로드

In [ ]:
from ultralytics import YOLO

print("YOLOv8L 모델 로딩 중...")
model = YOLO(MODEL_PATH)
print(f"모델 로드 완료! (conf={CONF}, iou={IOU}, imgsz={IMGSZ})")

## 3. 스냅샷 추출 (중간 프레임 + YOLO 시각화)

In [ ]:
import cv2
from tqdm.auto import tqdm

summary = []

for video_name, paths in VIDEO_SETS.items():
    input_dir  = paths["input"]
    output_dir = paths["output"]

    video_files = sorted([f for f in os.listdir(input_dir) if f.endswith('.mp4')])
    print(f"\n========== [{video_name}] {len(video_files)}개 클립 처리 시작 ==========")

    saved = 0
    no_detection = 0

    for video_filename in tqdm(video_files, desc=f"{video_name} 스냅샷"):
        video_path = os.path.join(input_dir, video_filename)

        # 중간 프레임 추출
        cap = cv2.VideoCapture(video_path)
        if not cap.isOpened():
            continue

        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        if total_frames <= 0:
            cap.release()
            continue

        cap.set(cv2.CAP_PROP_POS_FRAMES, total_frames // 2)
        ret, frame = cap.read()
        cap.release()

        if not ret or frame is None:
            continue

        # YOLO 추론 (MVP 15 클래스만 필터)
        results = model(
            frame,
            conf=CONF,
            iou=IOU,
            imgsz=IMGSZ,
            classes=MVP_CLASS_IDS,
            verbose=False
        )

        # 탐지 수 확인
        det_count = len(results[0].boxes)
        if det_count == 0:
            no_detection += 1

        # 바운딩박스 시각화 이미지 저장
        annotated = results[0].plot()
        out_filename = video_filename.replace(".mp4", ".jpg")
        out_path = os.path.join(output_dir, out_filename)
        cv2.imwrite(out_path, annotated)
        saved += 1

    print(f"  저장 완료: {saved}개 | 탐지 없음: {no_detection}개")
    summary.append({"영상": video_name, "총 클립": len(video_files),
                    "저장": saved, "탐지 없음": no_detection})

print("\n전체 스냅샷 추출 완료!")

## 4. 결과 요약

In [ ]:
import pandas as pd
from IPython.display import display

df_summary = pd.DataFrame(summary)
print("\n===== 스냅샷 추출 요약 =====")
display(df_summary)

for name, paths in VIDEO_SETS.items():
    n = len([f for f in os.listdir(paths["output"]) if f.endswith('.jpg')])
    print(f"[{name}] {paths['output']} → {n}개 이미지 저장됨")

## 5. 샘플 미리보기 (각 영상 5장)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import matplotlib
import os

# 한글 폰트 설정
os.system("apt-get install -y fonts-nanum > /dev/null 2>&1")
fm._load_fontmanager(try_read_cache=False)
nanum_fonts = [f.fname for f in fm.fontManager.ttflist if 'Nanum' in f.name]
if nanum_fonts:
    font_prop = fm.FontProperties(fname=nanum_fonts[0])
    matplotlib.rcParams['font.family'] = font_prop.get_name()
else:
    font_path = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"
    fm.fontManager.addfont(font_path)
    font_prop = fm.FontProperties(fname=font_path)
    matplotlib.rcParams['font.family'] = font_prop.get_name()
matplotlib.rcParams['axes.unicode_minus'] = False

PREVIEW_COUNT = 5

for video_name, paths in VIDEO_SETS.items():
    output_dir = paths["output"]
    imgs = sorted([f for f in os.listdir(output_dir) if f.endswith('.jpg')])

    # 균등 간격으로 5장 선택
    step = max(1, len(imgs) // PREVIEW_COUNT)
    sample_imgs = imgs[::step][:PREVIEW_COUNT]

    fig, axes = plt.subplots(1, len(sample_imgs), figsize=(22, 5))
    fig.suptitle(f'[{video_name}] 스냅샷 미리보기 (conf={CONF}, iou={IOU}, imgsz={IMGSZ})',
                 fontsize=13, fontweight='bold', fontproperties=font_prop)

    for ax, img_name in zip(axes, sample_imgs):
        img_path = os.path.join(output_dir, img_name)
        img = cv2.imread(img_path)
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        ax.imshow(img_rgb)
        ax.set_title(img_name[:20], fontsize=7)
        ax.axis('off')

    plt.tight_layout()
    plt.savefig(f'preview_{video_name}.png', dpi=120, bbox_inches='tight')
    plt.show()
    print(f"미리보기 저장: preview_{video_name}.png")

## 6. Drive에 백업

In [ ]:
import shutil

for video_name, paths in VIDEO_SETS.items():
    drive_out = paths["drive_output"]
    local_out = paths["output"]

    if os.path.exists(drive_out):
        shutil.rmtree(drive_out)
    shutil.copytree(local_out, drive_out)

    n = len([f for f in os.listdir(drive_out) if f.endswith('.jpg')])
    print(f"[{video_name}] Drive 백업 완료 → {drive_out} ({n}개)")

# 미리보기 이미지도 Drive에 저장
for video_name in VIDEO_SETS.keys():
    src = f'preview_{video_name}.png'
    dst = os.path.join(BASE_DRIVE, src)
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f"미리보기 Drive 저장: {dst}")

print("\nDrive 백업 완료!")